# Train YOLOv8n nhận diện xe ưu tiên (Nhóm 9 - ITS)
Đây là đúng quy trình nhóm đã chạy trên Google Colab (GPU T4). Dataset: Ambulance-Fire-Police Computer Vision Dataset (jaa, Roboflow Universe, v5, CC BY 4.0), tải định dạng YOLOv8 rồi đặt file zip vào Google Drive tại `MyDrive/ITS/`.
Chạy lần lượt từng ô sau khi bật Runtime → T4 GPU.

In [ ]:
# 1. Chuẩn bị: cài thư viện, nối Drive, giải nén, sửa data.yaml, chia lại train/valid/test (80/10/10 theo ảnh gốc)
!nvidia-smi -L
!pip install -q ultralytics

from google.colab import drive, files
drive.mount('/content/drive')
zip_path = '/content/drive/MyDrive/ITS/Ambulance-Fire-Police.v5i.yolov8.zip'

import os, glob, re, random, shutil, collections, yaml
!rm -rf /content/dataset && mkdir -p /content/dataset
!unzip -q "{zip_path}" -d /content/dataset
data_yaml = glob.glob('/content/dataset/**/data.yaml', recursive=True)[0]
root = os.path.dirname(data_yaml)
cfg = yaml.safe_load(open(data_yaml))
print('Cac lop:', cfg['names'])   # Ambulance, Fire truck, Police car
cfg['path'] = root
cfg['train'] = 'train/images'
cfg['val'] = 'valid/images'
cfg['test'] = 'test/images'
yaml.safe_dump(cfg, open(data_yaml, 'w'), allow_unicode=True)

# Bo ban dau chi co thu muc train, nen tu chia lai. Cac ban augment cua cung 1 anh goc
# (cung ten truoc ".rf.") phai nam chung 1 tap, tranh lot anh giong nhau sang tap test.
random.seed(42)
pool = []
for sub in ['train', 'valid', 'val', 'test']:
    pool += sorted(glob.glob(f'{root}/{sub}/images/*'))
groups = collections.defaultdict(list)
for p in pool:
    groups[re.split(r'\.rf\.', os.path.basename(p))[0]].append(p)
keys = sorted(groups)          # sorted de lan chay lai ra dung cung mot cach chia
random.shuffle(keys)
n = len(keys)
n_val, n_test = int(n * 0.1), int(n * 0.1)
split = {}
for i, k in enumerate(keys):
    s = 'valid' if i < n_val else 'test' if i < n_val + n_test else 'train'
    for p in groups[k]:
        split[p] = s
for p, s in split.items():
    src = p.split('/images/')[0]
    base = os.path.splitext(os.path.basename(p))[0]
    lbl = os.path.join(src, 'labels', base + '.txt')
    os.makedirs(f'{root}/{s}/images', exist_ok=True)
    os.makedirs(f'{root}/{s}/labels', exist_ok=True)
    if src != f'{root}/{s}':
        shutil.move(p, f'{root}/{s}/images/')
        if os.path.exists(lbl):
            shutil.move(lbl, f'{root}/{s}/labels/')
for s in ['train', 'valid', 'test']:
    print(s, '=', len(glob.glob(f'{root}/{s}/images/*')), 'anh')

# Luu ket qua train thang vao Drive de khong mat khi Colab ngat ket noi
!mkdir -p /content/drive/MyDrive/ITS/runs
!ln -sfn /content/drive/MyDrive/ITS/runs /content/runs

In [ ]:
# 2. Train 30 epoch tu yolov8n.pt (khoang 1 gio 10 phut tren T4)
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model.train(data=data_yaml, epochs=30, imgsz=640, batch=16, patience=10, project='/content/runs', name='its', exist_ok=True)

In [ ]:
# 3. Cham diem tren tap TEST (tap model chua tung thay)
best = YOLO('/content/runs/its/weights/best.pt')
m = best.val(data=data_yaml, split='test', imgsz=640, conf=0.001)
print('Precision TB:', m.box.mp)
print('Recall TB   :', m.box.mr)
print('mAP@0.5     :', m.box.map50)
print('mAP@0.5:0.95:', m.box.map)
for i, name in best.names.items():
    p, r, ap50, ap = m.box.class_result(i)
    print(f'{name}: P={p:.3f} R={r:.3f} mAP50={ap50:.3f}')

In [ ]:
# 4. Xuat ONNX (de gan vao web) va dong goi ket qua de tai ve
best.export(format='onnx', imgsz=640, opset=12, simplify=False)
!mkdir -p /content/ket_qua
!cp /content/runs/its/weights/best.pt /content/runs/its/weights/best.onnx /content/ket_qua/
!cp /content/runs/its/results.csv /content/runs/its/confusion_matrix.png /content/ket_qua/
!cp /content/runs/its/results.png /content/runs/its/val_batch0_labels.jpg /content/runs/its/val_batch0_pred.jpg /content/ket_qua/ 2>/dev/null
!cp {data_yaml} /content/ket_qua/data.yaml
!cd /content && zip -qr ket_qua.zip ket_qua
files.download('/content/ket_qua.zip')